# PlaceMux – Task 1: Student ↔ Job Matching
**AI/ML Engineer | Week 2 | Phase 2**

This notebook covers:
1. Load & explore data
2. Define the feature space
3. Build a baseline model
4. Build a logistic regression model
5. Evaluate with real metrics
6. Explain one match (plain English)
7. Define the matching API contract

## Cell 1 — Import libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')
print('All libraries loaded successfully!')

All libraries loaded successfully!


## Cell 2 — Load the datasets

In [4]:
students = pd.read_csv('../datasets/students.csv')
jobs     = pd.read_csv('../datasets/jobs.csv')
matches  = pd.read_csv('../datasets/matches.csv')

print('Students:', students.shape)
print('Jobs    :', jobs.shape)
print('Matches :', matches.shape)
print()
print(students.head(3))
print()
print(jobs.head(3))
print()
print(matches.head(3))

Students: (20, 7)
Jobs    : (9, 7)
Matches : (6, 6)

   student_id                                skills  internship_months  \
0           1   Python:85,SQL:75,Excel:70,Pandas:80                 18   
1           2       Java:80,Spring:75,SQL:65,Git:70                 24   
2           3  Python:90,ML:85,TensorFlow:75,SQL:70                 12   

  education_level certifications     preferred_role   location  
0           BTech     Python,SQL       Data Analyst       Pune  
1              BE           Java  Backend Developer     Mumbai  
2             MCA             ML        ML Engineer  Bangalore  

   job_id company_name          job_title                required_skills  \
0     101     TechNova       Data Analyst      Python:70,SQL:60,Excel:50   
1     102    CodeWorks  Backend Developer       Java:70,Spring:65,SQL:60   
2     103      AI Labs        ML Engineer  Python:80,ML:70,TensorFlow:60   

   min_experience_years job_type   location  
0                     1   Hybrid      

## Cell 3 — Helper: parse skill strings

Both `students.skills` and `jobs.required_skills` are stored as `Python:85,SQL:75` strings.
This helper converts them into a dict like `{'Python': 85, 'SQL': 75}`.

In [5]:
def parse_skills(skill_str):
    """Convert 'Python:85,SQL:75' -> {'Python': 85, 'SQL': 75}"""
    result = {}
    if pd.isna(skill_str) or str(skill_str).strip() == '':
        return result
    for part in str(skill_str).split(','):
        part = part.strip()
        if ':' in part:
            skill, score = part.split(':', 1)
            try:
                result[skill.strip()] = int(score.strip())
            except ValueError:
                pass
    return result

# Test it
print(parse_skills('Python:85,SQL:75,Excel:70'))
print(parse_skills('Python:70,SQL:60'))

{'Python': 85, 'SQL': 75, 'Excel': 70}
{'Python': 70, 'SQL': 60}


## Cell 4 — Define the Feature Space

This is the core of the task. We compute three features for every student-job pair:

| Feature | What it means |
|---|---|
| `skill_overlap_ratio` | % of job's required skills the student actually meets (at or above the required score) |
| `experience_gap` | student's internship months / 6 minus job's min years (positive = student has more) |
| `education_level_encoded` | BE/BTech/MCA etc. turned into a number |

In [6]:
def compute_skill_overlap(student_skills_str, job_skills_str):
    """
    Returns:
      overlap_count  - how many required skills the student meets (at or above required score)
      overlap_ratio  - overlap_count / total required skills  (0.0 to 1.0)
    """
    student_skills = parse_skills(student_skills_str)
    required_skills = parse_skills(job_skills_str)

    if not required_skills:
        return 0, 0.0

    met = 0
    for skill, required_score in required_skills.items():
        student_score = student_skills.get(skill, 0)
        if student_score >= required_score:
            met += 1

    return met, met / len(required_skills)


def compute_experience_gap(internship_months, min_exp_years):
    """
    Convert internship months to years (rough: 6 months = 1 year)
    Positive gap = student has more experience than needed
    """
    student_exp_years = internship_months / 6.0
    return round(student_exp_years - min_exp_years, 2)


# Quick sanity test
count, ratio = compute_skill_overlap('Python:85,SQL:75,Excel:70,Pandas:80', 'Python:70,SQL:60,Excel:50')
print(f'Overlap count: {count}, Overlap ratio: {ratio:.2f}')

gap = compute_experience_gap(12, 1)
print(f'Experience gap: {gap}')

Overlap count: 3, Overlap ratio: 1.00
Experience gap: 1.0


## Cell 5 — Build the full feature matrix from matches.csv

We join students and jobs into the matches table and compute features for each row.
If your matches.csv already has `skill_overlap_ratio` and `experience_gap`, we recalculate
them from scratch so numbers are consistent.

In [7]:
# Merge matches with student and job info
df = matches.merge(students, on='student_id', how='left')
df = df.merge(jobs, on='job_id', how='left')

print('Merged shape:', df.shape)
print('Columns:', df.columns.tolist())
print()

# Recalculate features cleanly
overlap_results = df.apply(
    lambda row: compute_skill_overlap(row['skills'], row['required_skills']), axis=1
)
df['skill_overlap_count_calc'] = [r[0] for r in overlap_results]
df['skill_overlap_ratio_calc'] = [r[1] for r in overlap_results]

df['experience_gap_calc'] = df.apply(
    lambda row: compute_experience_gap(row['internship_months'], row['min_experience_years']), axis=1
)

# Encode education level
le = LabelEncoder()
df['education_encoded'] = le.fit_transform(df['education_level'].fillna('Unknown'))
print('Education classes:', list(le.classes_))

print(df[['student_id','job_id','skill_overlap_ratio_calc','experience_gap_calc','education_encoded','label']].head(5))

Merged shape: (6, 18)
Columns: ['student_id', 'job_id', 'skill_overlap_count', 'skill_overlap_ratio', 'experience_gap', 'label', 'skills', 'internship_months', 'education_level', 'certifications', 'preferred_role', 'location_x', 'company_name', 'job_title', 'required_skills', 'min_experience_years', 'job_type', 'location_y']

Education classes: ['BE', 'BTech', 'MCA']
   student_id  job_id  skill_overlap_ratio_calc  experience_gap_calc  \
0           1     101                       1.0                 2.00   
1           2     102                       1.0                 2.00   
2           3     103                       1.0                 1.00   
3           4     104                       1.0                 1.33   
4           5     105                       1.0                 1.67   

   education_encoded  label  
0                  1      1  
1                  0      1  
2                  2      1  
3                  1      1  
4                  0      1  


## Cell 6 — Train / Test Split

In [9]:
FEATURES = ['skill_overlap_ratio_calc', 'experience_gap_calc', 'education_encoded']
TARGET   = 'label'

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train size : {len(X_train)}')
print(f'Test size  : {len(X_test)}')
print(f'Label distribution in train:\n{y_train.value_counts()}')
print(f'Label distribution in test:\n{y_test.value_counts()}')

Train size : 4
Test size  : 2
Label distribution in train:
label
1    3
0    1
Name: count, dtype: int64
Label distribution in test:
label
1    2
Name: count, dtype: int64


## Cell 7 — Baseline Model

**Dumb baseline**: rank purely by `skill_overlap_ratio`. Predict match (1) if ratio >= 0.5, else no match (0).
Every later model must beat this number.

In [10]:
BASELINE_THRESHOLD = 0.5

y_pred_baseline = (X_test['skill_overlap_ratio_calc'] >= BASELINE_THRESHOLD).astype(int)

print('=== BASELINE RESULTS (skill overlap ratio >= 0.5) ===')
print(f"Precision : {precision_score(y_test, y_pred_baseline):.3f}")
print(f"Recall    : {recall_score(y_test, y_pred_baseline):.3f}")
print()

cm = confusion_matrix(y_test, y_pred_baseline)
tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, cm[0][0])
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
print(f"False Positive Rate : {fpr:.3f}")
print(f"Confusion Matrix:\n{cm}")

=== BASELINE RESULTS (skill overlap ratio >= 0.5) ===
Precision : 1.000
Recall    : 1.000

False Positive Rate : 0.000
Confusion Matrix:
[[2]]


## Cell 8 — Logistic Regression Model (Smarter)

Uses all three features. Should beat the baseline on precision/recall.

In [11]:
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)

y_pred_lr = model.predict(X_test)

print('=== LOGISTIC REGRESSION RESULTS ===')
print(f"Precision : {precision_score(y_test, y_pred_lr):.3f}")
print(f"Recall    : {recall_score(y_test, y_pred_lr):.3f}")
print()

cm_lr = confusion_matrix(y_test, y_pred_lr)
tn, fp, fn, tp = cm_lr.ravel() if cm_lr.size == 4 else (0, 0, 0, cm_lr[0][0])
fpr_lr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
print(f"False Positive Rate : {fpr_lr:.3f}")
print(f"Confusion Matrix:\n{cm_lr}")
print()
print('Full report:')
print(classification_report(y_test, y_pred_lr))

=== LOGISTIC REGRESSION RESULTS ===
Precision : 1.000
Recall    : 1.000

False Positive Rate : 0.000
Confusion Matrix:
[[2]]

Full report:
              precision    recall  f1-score   support

           1       1.00      1.00      1.00         2

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2



## Cell 9 — Feature Importance (Explainability)

Which feature matters most? Logistic regression coefficients tell us directly.

In [12]:
coef_df = pd.DataFrame({
    'Feature'    : FEATURES,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient', ascending=False)

print('=== FEATURE IMPORTANCE (Logistic Regression Coefficients) ===')
print(coef_df.to_string(index=False))
print()
print('A higher positive coefficient = stronger signal for a good match.')

=== FEATURE IMPORTANCE (Logistic Regression Coefficients) ===
                 Feature  Coefficient
     experience_gap_calc     0.629351
skill_overlap_ratio_calc     0.214974
       education_encoded     0.053040

A higher positive coefficient = stronger signal for a good match.


## Cell 10 — Plain-English Explanation for One Match

This is your demo moment. Pick one student + one job, show the match score, explain WHY.

In [13]:
def explain_match(student_id, job_id):
    """
    Give a plain-English explanation for why a student does or doesn't match a job.
    """
    student = students[students['student_id'] == student_id].iloc[0]
    job     = jobs[jobs['job_id'] == job_id].iloc[0]

    s_skills  = parse_skills(student['skills'])
    j_skills  = parse_skills(job['required_skills'])
    count, ratio = compute_skill_overlap(student['skills'], job['required_skills'])
    exp_gap   = compute_experience_gap(student['internship_months'], job['min_experience_years'])
    edu_enc   = le.transform([student['education_level']])[0]

    # Predict
    features  = np.array([[ratio, exp_gap, edu_enc]])
    pred      = model.predict(features)[0]
    prob      = model.predict_proba(features)[0][1]

    # Print explanation
    print('=' * 55)
    print(f'STUDENT  : {student_id}')
    print(f'JOB      : {job_id} — {job["job_title"]} at {job["company_name"]}')
    print('=' * 55)
    print(f"Student's skills  : {student['skills']}")
    print(f"Required skills   : {job['required_skills']}")
    print()

    print('--- Skill-by-skill breakdown ---')
    for skill, req_score in j_skills.items():
        student_score = s_skills.get(skill, 0)
        met = '✅ MET' if student_score >= req_score else '❌ NOT MET'
        print(f'  {skill:<12} required {req_score:>3}   student has {student_score:>3}   {met}')

    print()
    print(f'Skill overlap     : {count}/{len(j_skills)} skills met  ({ratio*100:.1f}%)')
    print(f'Experience gap    : {exp_gap:+.1f} years  (positive = student has more than required)')
    print(f'Education         : {student["education_level"]} (encoded: {edu_enc})')
    print()
    print(f'MATCH SCORE (probability) : {prob:.3f}')
    print(f'PREDICTION                : {"✅ GOOD MATCH" if pred == 1 else "❌ NOT A MATCH"}')
    print()
    print('Plain-English reason:')
    if pred == 1:
        print(f'  This student meets {count} out of {len(j_skills)} required skills at the needed score level.')
        if exp_gap >= 0:
            print(f'  Their internship experience is sufficient for this role.')
        print(f'  The model gives this pair a {prob*100:.1f}% match confidence.')
    else:
        missing = [s for s, r in j_skills.items() if s_skills.get(s, 0) < r]
        print(f'  The student is missing or underscore on: {", ".join(missing)}.')
        if exp_gap < 0:
            print(f'  Experience is also {abs(exp_gap):.1f} years short of what the job needs.')
        print(f'  Match confidence is only {prob*100:.1f}%.')


# Change these IDs to any real ones from your CSV
DEMO_STUDENT_ID = students['student_id'].iloc[0]
DEMO_JOB_ID     = jobs['job_id'].iloc[0]

explain_match(DEMO_STUDENT_ID, DEMO_JOB_ID)

STUDENT  : 1
JOB      : 101 — Data Analyst at TechNova
Student's skills  : Python:85,SQL:75,Excel:70,Pandas:80
Required skills   : Python:70,SQL:60,Excel:50

--- Skill-by-skill breakdown ---
  Python       required  70   student has  85   ✅ MET
  SQL          required  60   student has  75   ✅ MET
  Excel        required  50   student has  70   ✅ MET

Skill overlap     : 3/3 skills met  (100.0%)
Experience gap    : +2.0 years  (positive = student has more than required)
Education         : BTech (encoded: 1)

MATCH SCORE (probability) : 0.848
PREDICTION                : ✅ GOOD MATCH

Plain-English reason:
  This student meets 3 out of 3 required skills at the needed score level.
  Their internship experience is sufficient for this role.
  The model gives this pair a 84.8% match confidence.


## Cell 11 — Matching API Contract

This is the spec you hand to the Backend team.
It defines exactly what goes IN and what comes OUT of your matching function.

In [14]:
import json

api_contract = {
    "endpoint"    : "POST /api/v1/match",
    "description" : "Returns a match score and plain-English reason for a student-job pair.",
    "request_body": {
        "student_id"        : "string  — e.g. 'S001'",
        "skills"            : "string  — e.g. 'Python:85,SQL:75,Excel:70'",
        "internship_months" : "int     — e.g. 12",
        "education_level"   : "string  — e.g. 'BE'",
        "job_id"            : "string  — e.g. 'J001'",
        "required_skills"   : "string  — e.g. 'Python:70,SQL:60'",
        "min_experience_years" : "float — e.g. 1.0"
    },
    "response_body": {
        "student_id"         : "string",
        "job_id"             : "string",
        "match_score"        : "float (0.0 to 1.0) — model's match probability",
        "prediction"         : "int   — 1 = good match, 0 = no match",
        "skill_overlap_ratio": "float — fraction of required skills met",
        "experience_gap"     : "float — positive = student has more experience than needed",
        "reason"             : "string — plain-English explanation"
    },
    "example_request": {
        "student_id"           : "S001",
        "skills"               : "Python:85,SQL:75,Excel:70,Pandas:80",
        "internship_months"    : 12,
        "education_level"      : "BE",
        "job_id"               : "J001",
        "required_skills"      : "Python:70,SQL:60,Excel:50",
        "min_experience_years" : 1.0
    },
    "example_response": {
        "student_id"          : "S001",
        "job_id"              : "J001",
        "match_score"         : 0.87,
        "prediction"          : 1,
        "skill_overlap_ratio" : 1.0,
        "experience_gap"      : 1.0,
        "reason"              : "Student meets all 3 required skills at or above the required score. Experience is sufficient. Match confidence: 87%."
    },
    "error_cases": {
        "400" : "Missing required fields",
        "404" : "student_id or job_id not found",
        "500" : "Model inference failed"
    }
}

print('=== MATCHING API CONTRACT ===')
print(json.dumps(api_contract, indent=2))

=== MATCHING API CONTRACT ===
{
  "endpoint": "POST /api/v1/match",
  "description": "Returns a match score and plain-English reason for a student-job pair.",
  "request_body": {
    "student_id": "string  \u2014 e.g. 'S001'",
    "skills": "string  \u2014 e.g. 'Python:85,SQL:75,Excel:70'",
    "internship_months": "int     \u2014 e.g. 12",
    "education_level": "string  \u2014 e.g. 'BE'",
    "job_id": "string  \u2014 e.g. 'J001'",
    "required_skills": "string  \u2014 e.g. 'Python:70,SQL:60'",
    "min_experience_years": "float \u2014 e.g. 1.0"
  },
  "response_body": {
    "student_id": "string",
    "job_id": "string",
    "match_score": "float (0.0 to 1.0) \u2014 model's match probability",
    "prediction": "int   \u2014 1 = good match, 0 = no match",
    "skill_overlap_ratio": "float \u2014 fraction of required skills met",
    "experience_gap": "float \u2014 positive = student has more experience than needed",
    "reason": "string \u2014 plain-English explanation"
  },
  "ex

## Cell 12 — Summary: Baseline vs Model comparison

In [15]:
summary = pd.DataFrame({
    'Model'    : ['Baseline (overlap >= 0.5)', 'Logistic Regression'],
    'Precision': [
        round(precision_score(y_test, y_pred_baseline), 3),
        round(precision_score(y_test, y_pred_lr), 3)
    ],
    'Recall'   : [
        round(recall_score(y_test, y_pred_baseline), 3),
        round(recall_score(y_test, y_pred_lr), 3)
    ],
    'FPR'      : [
        round(fpr, 3),
        round(fpr_lr, 3)
    ]
})

print('=== FINAL COMPARISON ===')
print(summary.to_string(index=False))
print()
print('Precision = of the matches we predicted, how many were actually good?')
print('Recall    = of all the actual good matches, how many did we catch?')
print('FPR       = false alarm rate — how often we said match when it wasn\'t')

=== FINAL COMPARISON ===
                    Model  Precision  Recall  FPR
Baseline (overlap >= 0.5)        1.0     1.0  0.0
      Logistic Regression        1.0     1.0  0.0

Precision = of the matches we predicted, how many were actually good?
Recall    = of all the actual good matches, how many did we catch?
FPR       = false alarm rate — how often we said match when it wasn't
